# Decision Tree Classifier
supervised machine learning algorithm used for classification tasks. It works by splitting the dataset into smaller and smaller subsets based on feature values, forming a tree-like structure of decisions. Each internal node represents a decision rule (based on a feature), each branch represents an outcome of that decision, and each leaf node represents a final class (prediction).

In [1]:
%pip install pandas scikit-learn matplotlib graphviz kagglehub opencv-python kagglehub

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohamedmaher5/vehicle-classification")

print("Path to dataset files:", path)

100%|██████████| 827M/827M [00:09<00:00, 90.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mohamedmaher5/vehicle-classification/versions/1


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [6]:
import os

print(os.listdir(os.path.join(path, "Vehicles")))

['Trains', 'Bikes', 'Cars', 'Motorcycles', 'Planes', 'Ships', 'Auto Rickshaws']


In [9]:
%pip install split-folders

In [10]:
import os
import splitfolders

# Original dataset
data_path = os.path.join(path, "Vehicles")

# New folder that will contain train/val/test
# output_path = os.path.join(path, "vehicles_split")
# For Colab
output_path = "/kaggle/working/vehicles_split"

# Split dataset: 80% train, 20% test
splitfolders.ratio(
    data_path,
    output=output_path,
    seed=42,
    ratio=(0.8, 0.0, 0.2),   # train, validation, test
    group_prefix=None
)

print("Dataset successfully split!")

Copying files: 5590 files [00:03, 1707.92 files/s]

Dataset successfully split!


In [11]:
import os
import cv2
import numpy as np

data_path = output_path  # the path printed by your kagglehub code

images = []
labels = []

IMG_SIZE = 64  # resize all images to 64x64

# Folder structure is usually /cats and /dogs
for category in ['Auto Rickshaws', 'Bikes', 'Cars', 'Motorcycles', 'Planes', 'Ships', 'Trains']:
    train_folder=os.path.join(data_path, 'train')
    folder = os.path.join(train_folder, category)

    if category == "Auto Rickshaws":
        label = 0
    elif category == "Bikes":
        label = 1
    elif category == "Cars":
        label = 2
    elif category == "Motorcycles":
        label = 3
    elif category == "Planes":
        label = 4
    elif category == "Ships":
        label = 5
    elif category == "Trains":
        label = 6

    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.flatten()  # convert image into 1D array
        images.append(img)
        labels.append(label)


for category in ['Auto Rickshaws', 'Bikes', 'Cars', 'Motorcycles', 'Planes', 'Ships', 'Trains']:
    train_folder=os.path.join(data_path, 'test')
    folder = os.path.join(train_folder, category)
    if category == "Auto Rickshaws":
        label = 0
    elif category == "Bikes":
        label = 1
    elif category == "Cars":
        label = 2
    elif category == "Motorcycles":
        label = 3
    elif category == "Planes":
        label = 4
    elif category == "Ships":
        label = 5
    elif category == "Trains":
        label = 6

    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.flatten()  # convert image into 1D array
        images.append(img)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Dataset loaded:", images.shape)


Dataset loaded: (5590, 12288)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

In [18]:
clf = DecisionTreeClassifier(max_depth=20)  # tune this
clf.fit(X_train, y_train)


DecisionTreeClassifier(max_depth=20)

In [19]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Decision Tree Accuracy:", acc)

Decision Tree Accuracy: 0.4338103756708408


In [15]:
for i in range(1, 51):
    clf = DecisionTreeClassifier(max_depth=i)  # tune this
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Iteration {i}: Decision Tree Accuracy: {acc}")

Iteration 1: Decision Tree Accuracy: 0.16815742397137745
Iteration 2: Decision Tree Accuracy: 0.22361359570661896
Iteration 3: Decision Tree Accuracy: 0.2898032200357782
Iteration 4: Decision Tree Accuracy: 0.3103756708407871
Iteration 5: Decision Tree Accuracy: 0.3470483005366726
Iteration 6: Decision Tree Accuracy: 0.3747763864042934
Iteration 7: Decision Tree Accuracy: 0.4141323792486583
Iteration 8: Decision Tree Accuracy: 0.40966010733452596


KeyboardInterrupt: 

In [21]:
import cv2

# Predict Single Image

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    print(img.flatten())
    img = img.flatten().reshape(1, -1)  # reshape for prediction

    pred = clf.predict(img)
    print(pred)
    if pred[0] == 0:
        return "Auto Rickshaws"
    elif pred[0] == 1:
        return "Bikes"
    elif pred[0] == 2:
        return "Cars"
    elif pred[0] == 3:
        return "Motorcycles"
    elif pred[0] == 4:
        return "Planes"
    elif pred[0] == 5:
        return "Ships"
    elif pred[0] == 6:
        return "Trains"
    else:
        return "Unknown"

test_image_path ="image.jpg"

predict_image(test_image_path)

[221 224 229 ... 236 230 232]
[3]


'Motorcycles'

In [22]:
# export the model
import pickle
pickle.dump(clf, open("vehicle_decision_tree_model.pkl", "wb"))